# SQLite Database and ERD

## Purpose
This notebook creates a simple relational database in SQLite using the cleaned BRFSS and CDC PLACES state-level datasets. It also supports the database design process by defining tables, explaining relationships, and running SQL queries that answer meaningful project questions.

## Why this notebook matters
A relational database helps organize the cleaned data in a structured way, supports reproducibility, and makes it easier to query behavioral and health outcome data together.

In [14]:
import pandas as pd
import sqlite3

In [15]:
BRFSS_state = pd.read_csv("../data/brfss_state.csv")
PLACES_state = pd.read_csv("../data/places_state_cleaned.csv")

In [16]:
BRFSS_state.head()

,StateAbbr,drnkdrv_cat2_pct,pneumo_yes_pct
0,AK,49.628959,20.235294
1,AL,38.743696,26.501605
2,AR,36.254906,28.686227
3,AZ,46.709870,29.519774
4,CA,46.025384,21.217435


In [17]:
PLACES_state.head()

,StateAbbr,Binge drinking among adults,Current cigarette smoking among adults,Diagnosed diabetes among adults,No leisure-time physical activity among adults,Obesity among adults
0,AK,17.837288,19.139286,10.960000,24.465000,35.573684
1,AL,13.949254,17.782308,15.358209,31.533582,41.870400
2,AR,14.590000,18.374667,13.860403,35.146000,40.967606
3,AZ,15.383333,14.355172,12.190000,25.060000,33.263333
4,CA,16.685217,12.511818,11.148246,24.531034,30.129630


## Previewing the State-Level Tables

Before loading the data into SQLite, both state-level tables are reviewed to confirm that they are clean, structured correctly, and ready for database use.

The `brfss_state` table contains behavioral response summaries, while the `places_state` table contains state-level public health outcome measures.

In [18]:
#create the SQLite connection
conn = sqlite3.connect("health_analysis.db")

In [19]:
BRFSS_state.to_sql("brfss_state", conn, if_exists="replace", index=False)
PLACES_state.to_sql("places_state", conn, if_exists="replace", index=False)

50

In [20]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,brfss_state
1,places_state


In [ ]:
#Preview Tables from SQL
pd.read_sql("SELECT * FROM brfss_state LIMIT 5;", conn)

,StateAbbr,drnkdrv_cat2_pct,pneumo_yes_pct
0,AK,49.628959,20.235294
1,AL,38.743696,26.501605
2,AR,36.254906,28.686227
3,AZ,46.709870,29.519774
4,CA,46.025384,21.217435


In [22]:
pd.read_sql("SELECT * FROM places_state LIMIT 5;", conn)


,StateAbbr,Binge drinking among adults,Current cigarette smoking among adults,Diagnosed diabetes among adults,No leisure-time physical activity among adults,Obesity among adults
0,AK,17.837288,19.139286,10.960000,24.465000,35.573684
1,AL,13.949254,17.782308,15.358209,31.533582,41.870400
2,AR,14.590000,18.374667,13.860403,35.146000,40.967606
3,AZ,15.383333,14.355172,12.190000,25.060000,33.263333
4,CA,16.685217,12.511818,11.148246,24.531034,30.129630


## SQL Query 1: Joining BRFSS and PLACES Data

This query combines behavioral data from BRFSS with health outcome data from CDC PLACES using the shared `StateAbbr` field.

This allows direct comparison between individual behaviors and population-level health outcomes.

In [23]:
query1 = """
SELECT
    b.StateAbbr,
    b.drnkdrv_cat2_pct,
    b.pneumo_yes_pct,
    p."Obesity among adults",
    p."Diagnosed diabetes among adults",
    p."No leisure-time physical activity among adults"
FROM brfss_state b
JOIN places_state p
    ON b.StateAbbr = p.StateAbbr
LIMIT 10;
"""

pd.read_sql(query1, conn)

,StateAbbr,drnkdrv_cat2_pct,pneumo_yes_pct,Obesity among adults,Diagnosed diabetes among adults,No leisure-time physical activity among adults
0,AK,49.628959,20.235294,35.573684,10.960000,24.465000
1,AL,38.743696,26.501605,41.870400,15.358209,31.533582
2,AR,36.254906,28.686227,40.967606,13.860403,35.146000
3,AZ,46.709870,29.519774,33.263333,12.190000,25.060000
4,CA,46.025384,21.217435,30.129630,11.148246,24.531034
5,CO,56.153934,20.904019,27.134921,9.615748,19.573810
6,CT,53.689085,23.176508,30.505556,9.238889,24.461111
7,DC,57.249766,22.700343,24.900000,8.150000,15.100000
8,DE,47.921532,29.962634,37.760000,12.266667,27.616667
9,FL,44.194644,27.989438,34.668217,12.824627,27.040299


### Interpretation

This query successfully joins the BRFSS and PLACES state-level tables using the shared `StateAbbr` field. The output confirms that both behavioral indicators and public health outcome measures can now be viewed together within the same dataset.

This is an important step in the project because it creates a structured way to compare behavioral patterns, such as drinking-related responses, with broader health outcomes like obesity, diabetes, and physical inactivity across states.

## SQL Query 2: States with the Highest Obesity Prevalence

This query ranks states by obesity prevalence using the PLACES table. It helps identify which states have the highest burden of obesity and creates a useful comparison point for the behavioral data from BRFSS.

In [24]:
query2 = """
SELECT
    StateAbbr,
    "Obesity among adults"
FROM places_state
ORDER BY "Obesity among adults" DESC
LIMIT 10;
"""

pd.read_sql(query2, conn)

,StateAbbr,Obesity among adults
0,MS,42.681935
1,LA,42.090984
2,AL,41.870400
3,WV,41.345631
4,AR,40.967606
5,OK,40.886301
6,TN,39.784358
7,IN,39.716763
8,NE,39.578977
9,IA,39.354011


### Interpretation

This query ranks the states with the highest obesity prevalence based on the PLACES dataset. The results show that several states, including Mississippi, Louisiana, and Alabama, have obesity rates above 40%, indicating a substantial chronic health burden.

This ranking is useful because it helps identify which states may require closer attention in public health analysis. It also provides a strong basis for comparing whether behavioral patterns from BRFSS align with these higher obesity levels.

## SQL Query 3: States with High Physical Inactivity and High Obesity

This query uses a grouped join and a HAVING clause to identify states where both physical inactivity and obesity are high. It helps test whether the pattern observed in the visual analysis also appears clearly in SQL.

In [25]:
query3 = """
SELECT
    b.StateAbbr,
    AVG(p."No leisure-time physical activity among adults") AS avg_inactivity,
    AVG(p."Obesity among adults") AS avg_obesity
FROM brfss_state b
JOIN places_state p
    ON b.StateAbbr = p.StateAbbr
GROUP BY b.StateAbbr
HAVING avg_inactivity > 30
   AND avg_obesity > 35
ORDER BY avg_obesity DESC;
"""

pd.read_sql(query3, conn)

,StateAbbr,avg_inactivity,avg_obesity
0,MS,36.438415,42.681935
1,LA,33.288281,42.090984
2,AL,31.533582,41.870400
3,WV,33.770909,41.345631
4,AR,35.146000,40.967606
5,OK,33.361039,40.886301
6,MO,31.914847,38.860465
7,GA,30.718590,38.442384
8,TX,30.738878,37.590722


### Interpretation

This query identifies states where both physical inactivity and obesity are high. The results strongly support the earlier visual analysis, which showed a clear positive relationship between these two variables.

States such as Mississippi, Louisiana, Alabama, and West Virginia appear at the top of the list, indicating that high inactivity and high obesity often occur together. This strengthens the conclusion that physical inactivity is an important behavioral factor associated with obesity at the state level.

## SQL Query 4: States with Obesity Above the National Average

This query uses a subquery to identify states whose obesity prevalence is higher than the overall average obesity prevalence across all states.

In [26]:
query4 = """
SELECT
    StateAbbr,
    "Obesity among adults"
FROM places_state
WHERE "Obesity among adults" >
    (
        SELECT AVG("Obesity among adults")
        FROM places_state
    )
ORDER BY "Obesity among adults" DESC;
"""

pd.read_sql(query4, conn)

,StateAbbr,Obesity among adults
0,MS,42.681935
1,LA,42.090984
2,AL,41.870400
3,WV,41.345631
4,AR,40.967606
5,OK,40.886301
6,TN,39.784358
7,IN,39.716763
8,NE,39.578977
9,IA,39.354011


### Interpretation

This query uses a subquery to identify states with obesity prevalence above the overall average across all states. The results show that a substantial number of states exceed this benchmark, especially in the South and Midwest.

This query is useful because it moves beyond simple ranking and introduces a broader comparison point. Instead of only identifying the highest states, it highlights which states are above the national average and may therefore represent a greater public health concern.

## Conclusion

In this notebook, a simple SQLite database was created using the cleaned BRFSS and CDC PLACES state-level datasets. Two tables were loaded into SQLite and linked through the shared `StateAbbr` field.

The SQL queries showed that the two datasets can be joined successfully and used to answer meaningful public health questions. The results highlighted states with the highest obesity prevalence, identified states where both physical inactivity and obesity are high, and used a subquery to compare obesity prevalence against the overall state average.

Together, these findings support the earlier visual analysis and show that SQL can be used not only to organize data, but also to strengthen the evidence behind the project’s main conclusions.